# Second half of the AO3 practical session

## This will focus on *scraping your own data* 

This session will address the following questions:
- how to use Selenium to scrape data 

- how to organize your data and make it easily analyzable 

### Exercise: Scrape metadata (25 min)

Below you will find the code for scraping and saving the data for 1 work id. Repeat as many times as you need. Afterwards we'll load the data and turn it into a dataframe that makes analysis easy. 

We will use the package Selenium and use a Chrome driver to fetch the data. Make sure to install Chrome before starting.

In [ ]:
# specify the id for the work you want to scrape
work_id = "73805911"


In [ ]:
# setup url 
# ?view_full_work=true means the whole text is loaded - useful for scraping the text data 
url = ("https://archiveofourown.org/works/" + work_id + "?view_full_work=true")

In [ ]:
# load relevant libraries 
from bs4 import BeautifulSoup
import pandas as pd

import time
from pathlib import Path
import json

import utils

In [ ]:
# setup chrome driver and get the html of the webpage
driver = utils.init_chrome_driver()
driver.get(url)

utils.tos_check(driver)
time.sleep(1)
driver.refresh()

utils.mature_check(driver)

html = driver.page_source

driver.quit()

In [ ]:
# make the html into a "Beautiful Soup" - makes life easy

soup = BeautifulSoup(html, "html.parser")

In [ ]:
# check the html was scraped correctly
title = soup.find("h2").get_text(strip = True)
title

In [ ]:
# using selectors to extract data 
# here are the most basic ones
# if you want, you can inspect the source code on AO3's website and see if you can develop selectors for the rest

author = soup.select_one("h3.byline.heading").get_text(strip = True)

language = soup.select_one("dd.language").get_text(strip = True)

In [ ]:

published_date = soup.select_one(f"dd.published").get_text()

words = soup.select_one(f"dd.words").get_text().replace(",", "")

In [ ]:
try:
    kudos = int(soup.select_one(f"dd.kudos").get_text().replace(",", ""))
except (AttributeError, ValueError):
    # this catches cases where kudos = 0
    kudos = 0

In [ ]:
try:
    hits = int(soup.select_one(f"dd.hits").get_text().replace(",", ""))
except (AttributeError, ValueError):
    # this catches cases where hits = 0
    hits = 0

In [ ]:
# using a utils function for getting the list of tags for each tag type 

rating = utils.get_tags(soup, "rating")
category = utils.get_tags(soup, "category")
fandom_tags = utils.get_tags(soup, "fandom")
freeform_tags = utils.get_tags(soup, "freeform")

In [ ]:
# you can check the values of the variables here
freeform_tags

In [ ]:
# optional - get the text. I would only scrape the text if you need it for your RQ
full_text = utils.get_fulltext(soup)

In [ ]:
# saving data
# I find the easiest method is to save all the data in a dictionary and then save that as a .json file in a data/ folder

work_dict = {
    "url": url,
    "work_id": work_id,
    "title": title,
    "author": author,
    "language": language,
    "published_date": published_date,
    "words": words,
    "kudos": kudos,
    "hits": hits,
    "rating": rating,
    "category": category,
    "fandom_tags": fandom_tags,
    "freeform_tags": freeform_tags
}


In [ ]:
# optional - if you scraped the text, add it here
work_dict['full_text'] = full_text

In [ ]:
# setup the output path
out_folder = Path("data/")
out_folder.mkdir(exist_ok=True)

In [ ]:
# save the dictionary with data
with open(out_folder.joinpath(f"{work_id}_metadata.json"), "w") as f:
    f.write(json.dumps(work_dict))

In [ ]:
# because then we can load all .json files in the data/ folder like this
json_files = list(out_folder.rglob("*.json"))

all_meta = []

for file in json_files:
    with open(file, "r") as f:
        content = json.load(f)
    all_meta.append(content)



In [ ]:
# and then make a pandas dataframe of it
df = pd.DataFrame(all_meta)

In [ ]:
# look at the df
df.head()

### Catch-up in plenum (5 min)

